# Clase 1 — Fundamentos de Criptografía y Teoría de la Información

**Curso:** Criptografía Aplicada  
**Temas:** Historia de la criptografía · Principios de Kerckhoffs · Entropía · Aleatoriedad · Cifrados clásicos · Análisis de frecuencia  

---

> *"La criptografía es el arte y la ciencia de mantener mensajes seguros."*  
> — Bruce Schneier, *Applied Cryptography* (1996)


## Tabla de Contenidos

1. [Historia de la Criptografía](#1-historia)
2. [Conceptos Fundamentales](#2-conceptos)
3. [Principios de Kerckhoffs](#3-kerckhoffs)
4. [Teoría de la Información y Entropía](#4-entropia)
5. [Aleatoriedad y Números Pseudoaleatorios](#5-aleatoriedad)
6. [Cifrado César](#6-cesar)
7. [Cifrado Vigenère](#7-vigenere)
8. [Cifrado Playfair](#8-playfair)
9. [Análisis de Frecuencia](#9-frecuencia)
10. [Práctica: Romper Cifrados Clásicos](#10-practica)
11. [Resumen y Conclusiones](#11-resumen)
12. [Referencias](#12-referencias)


In [ ]:
# Importaciones necesarias para toda la clase
import string
import math
import random
import collections
import itertools
from typing import List, Tuple, Dict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
print('Librerías cargadas correctamente.')

---
## 1. Historia de la Criptografía <a id='1-historia'></a>

La criptografía es tan antigua como la escritura misma. A lo largo de la historia, la necesidad de comunicar información de manera secreta ha impulsado el desarrollo de técnicas cada vez más sofisticadas.

### 1.1 Antigüedad (≈ 3000 a.C. – 500 d.C.)

| Época | Ejemplo | Descripción |
|-------|---------|-------------|
| ~1900 a.C. | Jeroglíficos no estándar (Egipto) | Sustitución de símbolos para mayor solemnidad |
| ~600 a.C. | **Escítala espartana** | Primer dispositivo criptográfico: transposición mediante cilindro |
| ~100 a.C. | **Cifrado César** | Desplazamiento del alfabeto; usado por Julio César |
| ~200 d.C. | **Atbash** (hebreo) | Sustitución inversa del alfabeto |

### 1.2 Edad Media y Renacimiento (500 – 1700)

- **Al-Kindi (≈ 850 d.C.):** Escribió *Risalah fi Istikhraj al-Mu'amma* (Manuscrito para el descifrado de mensajes cifrados), el primer tratado conocido sobre **análisis de frecuencia**.
- **León Battista Alberti (1467):** Inventó el **disco de cifrado** y propuso el concepto de cifrado polialfabético.
- **Johannes Trithemius (1518):** *Polygraphia*, primer libro impreso sobre criptografía.
- **Blaise de Vigenère (1586):** Publicó el cifrado polialfabético conocido como **cifrado Vigenère** (aunque basado en el trabajo previo de Bellaso).

### 1.3 Era Moderna (1700 – 1949)

- **Thomas Jefferson (≈ 1795):** Rueda de cifrado con 26 discos.
- **Auguste Kerckhoffs (1883):** Publicó sus famosos **seis principios** sobre seguridad criptográfica.
- **Primera y Segunda Guerra Mundial:** Uso masivo de máquinas cifradoras como **ENIGMA** (Alemania) y su ruptura por Alan Turing en Bletchley Park.
- **Claude Shannon (1949):** Publicó *"Communication Theory of Secrecy Systems"*, fundando la **teoría matemática** de la criptografía.

### 1.4 Criptografía Moderna (1949 – presente)

- **1976:** Whitfield Diffie y Martin Hellman publican *"New Directions in Cryptography"* — criptografía de clave pública.
- **1977:** RSA (Rivest, Shamir, Adleman) — primer sistema práctico de clave pública.
- **1977:** DES (Data Encryption Standard) adoptado por el NIST.
- **2001:** AES (Advanced Encryption Standard) reemplaza a DES.
- **Hoy:** Criptografía postcuántica, pruebas de conocimiento cero, criptomonedas.


In [ ]:
# Línea de tiempo interactiva de la criptografía
eventos = [
    (-1900, 'Jeroglíficos\nnо estándar'),
    (-600,  'Escítala\nespartana'),
    (-100,  'Cifrado\nCésar'),
    (850,   'Al-Kindi\n(análisis frec.)'),
    (1467,  'Alberti\n(disco cifrado)'),
    (1586,  'Vigenère'),
    (1883,  'Kerckhoffs'),
    (1943,  'Ruptura\nENIGMA'),
    (1949,  'Shannon'),
    (1976,  'Diffie-Hellman'),
    (1977,  'RSA / DES'),
    (2001,  'AES'),
]

fig, ax = plt.subplots(figsize=(16, 4))
años = [e[0] for e in eventos]
labels = [e[1] for e in eventos]

ax.hlines(0, min(años)-50, max(años)+50, colors='steelblue', linewidth=2)
for i, (año, label) in enumerate(eventos):
    offset = 0.35 if i % 2 == 0 else -0.55
    ax.plot(año, 0, 'o', color='tomato', markersize=9, zorder=5)
    ax.text(año, offset, f'{año}\n{label}', ha='center', va='bottom' if offset > 0 else 'top',
            fontsize=8, color='#222')

ax.set_xlim(min(años)-200, max(años)+100)
ax.set_ylim(-1.2, 1.0)
ax.axis('off')
ax.set_title('Línea de Tiempo de la Criptografía', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Conceptos Fundamentales <a id='2-conceptos'></a>

### 2.1 Terminología básica

| Término | Definición |
|---------|------------|
| **Texto plano (plaintext)** | Mensaje original legible |
| **Texto cifrado (ciphertext)** | Mensaje cifrado, ilegible sin la clave |
| **Cifrado (cipher)** | Algoritmo de transformación |
| **Clave (key)** | Parámetro secreto que controla el cifrado |
| **Cifrar (encrypt)** | Transformar plaintext → ciphertext |
| **Descifrar (decrypt)** | Transformar ciphertext → plaintext |
| **Criptoanálisis** | Arte de romper cifrados sin conocer la clave |
| **Criptología** | Disciplina que abarca criptografía + criptoanálisis |

### 2.2 Modelo general de comunicación segura

```
Alice                   Canal inseguro                Bob
  │                          │                          │
  │ plaintext (m)            │                          │
  │──────────────────────────│──────────────────────────│
  │     Enc(m, k) = c        │       Dec(c, k) = m      │
  │                          │                          │
  │                     Eve (atacante)                  │
  │                     observa 'c'                     │
```

### 2.3 Tipos de ataques criptoanalíticos

| Tipo de ataque | Información disponible para Eve |
|----------------|---------------------------------|
| **Solo texto cifrado (COA)** | Solo el ciphertext |
| **Texto plano conocido (KPA)** | Pares (plaintext, ciphertext) |
| **Texto plano elegido (CPA)** | Puede obtener cifrado de plaintexts elegidos |
| **Texto cifrado elegido (CCA)** | Puede obtener descifrado de ciphertexts elegidos |

### 2.4 Propiedades de seguridad

- **Confidencialidad:** Solo el receptor autorizado puede leer el mensaje.
- **Integridad:** El mensaje no fue alterado en tránsito.
- **Autenticidad:** El mensaje proviene realmente del remitente declarado.
- **No repudio:** El remitente no puede negar haber enviado el mensaje.


---
## 3. Principios de Kerckhoffs <a id='3-kerckhoffs'></a>

Auguste Kerckhoffs von Nieuwenhof publicó en 1883 el artículo *"La cryptographie militaire"* en el *Journal des sciences militaires*. En él enunció **seis principios** que todo sistema criptográfico militar debería cumplir:

### Los Seis Principios

| # | Principio | Significado práctico |
|---|-----------|---------------------|
| 1 | **El sistema debe ser, si no teóricamente irrompible, prácticamente irrompible** | La seguridad computacional es suficiente |
| 2 | **El sistema no debe requerir secreto; puede caer en manos del enemigo sin inconveniente** | 🔑 **La seguridad reside en la CLAVE, no en el algoritmo** |
| 3 | **La clave debe ser comunicable y retenible sin notas** | Usabilidad práctica |
| 4 | **El criptograma debe ser transmisible por telégrafo** | Compatibilidad con infraestructura existente |
| 5 | **El aparato o documentos deben ser portátiles y manejables por una sola persona** | Portabilidad |
| 6 | **El sistema debe ser fácil de usar** | No debe requerir conocimiento de larga lista de reglas |

### El Principio más Importante (Principio 2)

> **"Seguridad a través de la oscuridad" (Security through Obscurity) es una mala práctica.**

El principio 2 implica que:
- Los algoritmos criptográficos modernos (AES, RSA, etc.) son **públicos y abiertos** al escrutinio.
- La fortaleza del sistema descansa **exclusivamente en la clave**.
- Esto permite que la comunidad científica analice y valide los algoritmos.

**Contraejemplo histórico:** El algoritmo A5/1 usado en GSM (telefonía móvil) fue mantenido en secreto. Cuando fue filtrado y analizado, resultó tener serias debilidades.

### Máxima de Shannon (relacionada)

> *"The enemy knows the system."*  
> — Claude Shannon, 1949


---
## 4. Teoría de la Información y Entropía <a id='4-entropia'></a>

Claude Shannon fundó la **Teoría de la Información** en 1948 con su artículo *"A Mathematical Theory of Communication"*. En 1949 aplicó estos conceptos a la criptografía.

### 4.1 Entropía de Shannon

La **entropía** $H(X)$ mide la **incertidumbre** o **información** de una variable aleatoria $X$ con distribución de probabilidad $P$:

$$H(X) = -\sum_{x \in \mathcal{X}} P(x) \log_2 P(x)$$

**Unidad:** bits (cuando se usa $\log_2$)

**Interpretación criptográfica:**
- **Alta entropía** → Distribución uniforme → Más impredecible → Más seguro
- **Baja entropía** → Distribución sesgada → Más predecible → Más vulnerable

### 4.2 Propiedades de la Entropía

- $H(X) \geq 0$ (siempre no-negativa)
- $H(X) \leq \log_2 |\mathcal{X}|$ (máxima cuando distribución es uniforme)
- $H(X) = 0$ solo cuando el resultado es determinista

### 4.3 Distancia de unicidad

Shannon también definió la **distancia de unicidad** $n_0$: el número mínimo de caracteres de texto cifrado necesarios para determinar de forma única la clave:

$$n_0 \approx \frac{H(K)}{D}$$

donde $H(K)$ es la entropía de la clave y $D$ es la **redundancia del lenguaje** ($D = \log_2 26 - H_{\text{idioma}} \approx 3.4$ bits/letra para el inglés).


In [ ]:
def entropia_shannon(probabilidades: List[float]) -> float:
    """Calcula la entropía de Shannon en bits."""
    return -sum(p * math.log2(p) for p in probabilidades if p > 0)

# Ejemplo 1: Moneda justa
H_moneda_justa = entropia_shannon([0.5, 0.5])
print(f'Moneda justa: H = {H_moneda_justa:.4f} bits')

# Ejemplo 2: Moneda sesgada
H_moneda_sesgada = entropia_shannon([0.9, 0.1])
print(f'Moneda sesgada (90/10): H = {H_moneda_sesgada:.4f} bits')

# Ejemplo 3: Dado justo de 6 caras
H_dado = entropia_shannon([1/6] * 6)
print(f'Dado justo (6 caras): H = {H_dado:.4f} bits')

# Ejemplo 4: Alfabeto inglés uniforme (26 letras)
H_alfabeto_uniforme = entropia_shannon([1/26] * 26)
print(f'Alfabeto uniforme (26 letras): H = {H_alfabeto_uniforme:.4f} bits')

# Ejemplo 5: Frecuencias reales del inglés
freq_ingles = [
    0.0817, 0.0149, 0.0278, 0.0425, 0.1270, 0.0223, 0.0202, 0.0609,
    0.0697, 0.0015, 0.0077, 0.0403, 0.0241, 0.0675, 0.0751, 0.0193,
    0.0010, 0.0599, 0.0633, 0.0906, 0.0276, 0.0098, 0.0236, 0.0015,
    0.0197, 0.0007
]
H_ingles = entropia_shannon(freq_ingles)
print(f'Inglés real (distribución de letras): H = {H_ingles:.4f} bits')
print(f'Redundancia del inglés: D ≈ {H_alfabeto_uniforme - H_ingles:.4f} bits/letra')

In [ ]:
# Visualización: Entropía vs. sesgo de probabilidad
p_values = np.linspace(0.001, 0.999, 1000)
H_values = [-p * np.log2(p) - (1-p) * np.log2(1-p) for p in p_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica izquierda: Entropía binaria
axes[0].plot(p_values, H_values, color='steelblue', linewidth=2)
axes[0].axvline(x=0.5, color='tomato', linestyle='--', label='Máximo en p=0.5')
axes[0].axhline(y=1.0, color='gray', linestyle=':', alpha=0.6)
axes[0].set_xlabel('Probabilidad p')
axes[0].set_ylabel('Entropía H(X) [bits]')
axes[0].set_title('Entropía de una distribución binaria')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfica derecha: Comparación de distribuciones de letras
letras = list(string.ascii_uppercase)
x_pos = np.arange(len(letras))
freq_uniforme = [1/26] * 26

axes[1].bar(x_pos - 0.2, freq_ingles, 0.4, label=f'Inglés (H={H_ingles:.2f} bits)', 
            color='steelblue', alpha=0.8)
axes[1].bar(x_pos + 0.2, freq_uniforme, 0.4, label=f'Uniforme (H={H_alfabeto_uniforme:.2f} bits)', 
            color='tomato', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(letras)
axes[1].set_xlabel('Letra')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Frecuencias de letras: Inglés real vs. Uniforme')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 4.4 Perfecto Secreto (Perfect Secrecy)

Shannon definió el **perfecto secreto** como la propiedad en la que el ciphertext no revela ninguna información sobre el plaintext:

$$P(M = m \mid C = c) = P(M = m) \quad \forall m, c$$

**Teorema de Shannon:** Para lograr perfecto secreto, el espacio de claves debe ser al menos tan grande como el espacio de mensajes ($|\mathcal{K}| \geq |\mathcal{M}|$).

El **One-Time Pad (OTP)** es el único sistema que logra perfecto secreto bajo condiciones ideales:
- Clave aleatoria de la misma longitud que el mensaje
- Clave usada solo una vez
- Clave destruida después del uso


---
## 5. Aleatoriedad y Números Pseudoaleatorios <a id='5-aleatoriedad'></a>

### 5.1 Importancia de la aleatoriedad en criptografía

La aleatoriedad es fundamental para:
- Generación de claves
- Valores nonce e IV (Initialization Vectors)
- Padding criptográfico
- Protocolos de intercambio de claves

### 5.2 Tipos de generadores

| Tipo | Descripción | Ejemplo | ¿Criptográficamente seguro? |
|------|-------------|---------|-----------------------------|
| **TRNG** (True RNG) | Basado en fenómenos físicos | Ruido térmico, radiación | ✅ Sí |
| **PRNG** (Pseudo RNG) | Algoritmo determinista con semilla | Mersenne Twister | ❌ No |
| **CSPRNG** (Cryptographically Secure PRNG) | PRNG con propiedades de seguridad adicionales | `/dev/urandom`, `secrets` de Python | ✅ Sí |

### 5.3 Propiedades de un buen CSPRNG

1. **Impredecibilidad hacia adelante:** Dado el estado actual, no se puede predecir la salida futura.
2. **Impredecibilidad hacia atrás:** Dado el estado actual, no se puede inferir el estado anterior.
3. **Superación de pruebas estadísticas:** NIST SP 800-22 define pruebas formales.

### 5.4 Generadores Lineales Congruenciales (LCG)

El LCG es el PRNG más simple (y más débil):

$$X_{n+1} = (a \cdot X_n + c) \mod m$$

- $m$ = módulo
- $a$ = multiplicador  
- $c$ = incremento
- $X_0$ = semilla

**Advertencia:** ¡Nunca usar LCG para aplicaciones criptográficas!


In [ ]:
class LCG:
    """Generador Lineal Congruencial — SOLO para fines didácticos.
    NO usar en aplicaciones criptográficas reales.
    """
    def __init__(self, seed: int, a: int = 1664525, c: int = 1013904223, m: int = 2**32):
        self.state = seed
        self.a = a
        self.c = c
        self.m = m

    def next(self) -> int:
        self.state = (self.a * self.state + self.c) % self.m
        return self.state

    def random_float(self) -> float:
        return self.next() / self.m

# Demostración: predictibilidad del LCG
lcg1 = LCG(seed=42)
lcg2 = LCG(seed=42)  # Misma semilla

secuencia1 = [lcg1.next() for _ in range(10)]
secuencia2 = [lcg2.next() for _ in range(10)]

print('LCG con semilla=42:')
print('Instancia 1:', secuencia1[:5])
print('Instancia 2:', secuencia2[:5])
print('¿Idénticas?:', secuencia1 == secuencia2)
print('\n⚠️  MISMA SEMILLA → MISMA SECUENCIA → PREDECIBLE → ¡INSEGURO!')

In [ ]:
import secrets

# Comparación visual: LCG vs CSPRNG
n_puntos = 10000

lcg = LCG(seed=12345)
lcg_x = [lcg.random_float() for _ in range(n_puntos)]
lcg_y = [lcg.random_float() for _ in range(n_puntos)]

csprng_x = [secrets.randbelow(10**9) / 10**9 for _ in range(n_puntos)]
csprng_y = [secrets.randbelow(10**9) / 10**9 for _ in range(n_puntos)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(lcg_x, lcg_y, s=0.5, alpha=0.5, color='steelblue')
axes[0].set_title('LCG (PRNG débil)\n¿Ves patrones?', fontsize=12)
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
axes[0].set_aspect('equal')

axes[1].scatter(csprng_x, csprng_y, s=0.5, alpha=0.5, color='tomato')
axes[1].set_title('CSPRNG (secrets)\nDistribución uniforme', fontsize=12)
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
axes[1].set_aspect('equal')

plt.suptitle('Comparación: PRNG débil vs. CSPRNG', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Demostración: Generación segura de claves con Python
import secrets
import os

# Clave de 256 bits (32 bytes) para AES-256
clave_256_bits = secrets.token_bytes(32)
print(f'Clave AES-256 (hex): {clave_256_bits.hex()}')
print(f'Longitud: {len(clave_256_bits)} bytes = {len(clave_256_bits) * 8} bits')

# Token URL-safe para uso en APIs
token_api = secrets.token_urlsafe(32)
print(f'\nToken API (URL-safe): {token_api}')

# Contraseña aleatoria
alfabeto = string.ascii_letters + string.digits + '!@#$%^&*'
contrasena = ''.join(secrets.choice(alfabeto) for _ in range(16))
print(f'\nContraseña aleatoria: {contrasena}')

# Entropía de la contraseña
H_contrasena = math.log2(len(alfabeto)) * 16
print(f'Entropía estimada: {H_contrasena:.1f} bits')

---
## 6. Cifrado César <a id='6-cesar'></a>

### 6.1 Historia y descripción

El **Cifrado César** (también llamado *ROT-n* o cifrado de desplazamiento) fue usado por Julio César para comunicar mensajes militares. Consiste en desplazar cada letra del alfabeto un número fijo de posiciones.

Suetonio describe en *De Vita Caesarum*:
> *"Si quería escribir algo secreto, lo escribía en clave, es decir, cambiando el orden de las letras del alfabeto, de modo que ninguna palabra pudiera ser comprendida. Si alguien desea descifrarlas y comprenderlas, debe sustituir la cuarta letra del alfabeto, es decir, D por A, y así sucesivamente con las demás."*

### 6.2 Descripción matemática

- **Cifrar:** $E(x) = (x + k) \mod 26$
- **Descifrar:** $D(x) = (x - k) \mod 26$

donde $x$ es la posición de la letra (A=0, B=1, ..., Z=25) y $k$ es la clave de desplazamiento.

**Espacio de claves:** Solo 25 claves posibles (≈ 4.6 bits de seguridad). ¡Trivialmente rompible!

### 6.3 Ejemplo

Con $k = 3$ (el que usaba César):

```
Plaintext:   A B C D E F G H I J K L M N O P Q R S T U V W X Y Z
Ciphertext:  D E F G H I J K L M N O P Q R S T U V W X Y Z A B C
```

| Plaintext | HOLA MUNDO |
|-----------|------------|
| Ciphertext | KROD PXQGR |


In [ ]:
def cifrar_cesar(texto: str, clave: int) -> str:
    """Cifra un texto usando el cifrado César."""
    resultado = []
    for char in texto.upper():
        if char.isalpha():
            resultado.append(chr((ord(char) - ord('A') + clave) % 26 + ord('A')))
        else:
            resultado.append(char)
    return ''.join(resultado)

def descifrar_cesar(texto: str, clave: int) -> str:
    """Descifra un texto cifrado con César."""
    return cifrar_cesar(texto, -clave)

# Ejemplo básico
mensaje = 'HOLA MUNDO'
clave = 3
cifrado = cifrar_cesar(mensaje, clave)
descifrado = descifrar_cesar(cifrado, clave)

print(f'Mensaje original: {mensaje}')
print(f'Clave de César:   {clave}')
print(f'Texto cifrado:    {cifrado}')
print(f'Texto descifrado: {descifrado}')

# Mostrar todas las rotaciones posibles
print('\nTodas las rotaciones posibles (ROT-0 a ROT-25):')
mensaje_largo = 'LACRIPTOGRAFIAESMARAVILLOSA'
for k in range(26):
    print(f'ROT-{k:2d}: {cifrar_cesar(mensaje_largo, k)}')

In [ ]:
def ataque_fuerza_bruta_cesar(ciphertext: str) -> List[Tuple[int, str]]:
    """Rompe el cifrado César por fuerza bruta (solo 25 posibilidades)."""
    resultados = []
    for k in range(1, 26):
        plaintext = descifrar_cesar(ciphertext, k)
        resultados.append((k, plaintext))
    return resultados

mensaje_secreto = cifrar_cesar('VENI VIDI VICI', 13)
print(f'Ciphertext: {mensaje_secreto}\n')
print('Ataque por fuerza bruta:')
for k, texto in ataque_fuerza_bruta_cesar(mensaje_secreto):
    print(f'  k={k:2d}: {texto}')

print(f'\n✅ La clave correcta es 13 (ROT-13). ¡Texto: VENI VIDI VICI!')

---
## 7. Cifrado Vigenère <a id='7-vigenere'></a>

### 7.1 Historia

El cifrado Vigenère fue descrito por Giovan Battista Bellaso en 1553 y atribuido erróneamente a Blaise de Vigenère. Durante tres siglos fue conocido como *"le chiffre indéchiffrable"* (el cifrado indescifrable). En 1863, **Friedrich Kasiski** publicó el primer método general para romperlo.

### 7.2 Descripción

El cifrado Vigenère es un **cifrado polialfabético**: usa múltiples alfabetos de sustitución definidos por una clave textual repetida.

- **Cifrar:** $C_i = (P_i + K_{i \mod |K|}) \mod 26$
- **Descifrar:** $P_i = (C_i - K_{i \mod |K|}) \mod 26$

donde $P_i$ es la $i$-ésima letra del plaintext y $K_j$ es el valor numérico de la $j$-ésima letra de la clave.

### 7.3 Tabla de Vigenère (Tabula Recta)

La tabla de Vigenère contiene 26 filas, cada una con el alfabeto desplazado una posición.


In [ ]:
def imprimir_tabla_vigenere():
    """Imprime la tabla de Vigenère (Tabula Recta)."""
    print('   ', end='')
    for c in string.ascii_uppercase:
        print(f' {c}', end='')
    print()
    print('   ' + '─' * 53)
    for i, row_key in enumerate(string.ascii_uppercase):
        print(f'{row_key} │', end='')
        for j in range(26):
            print(f' {chr((i + j) % 26 + ord("A"))}', end='')
        print()

imprimir_tabla_vigenere()

In [ ]:
def cifrar_vigenere(texto: str, clave: str) -> str:
    """Cifra texto usando el cifrado Vigenère."""
    texto = texto.upper()
    clave = clave.upper()
    resultado = []
    j = 0  # índice de la clave (solo avanza con letras)
    for char in texto:
        if char.isalpha():
            desplazamiento = ord(clave[j % len(clave)]) - ord('A')
            resultado.append(chr((ord(char) - ord('A') + desplazamiento) % 26 + ord('A')))
            j += 1
        else:
            resultado.append(char)
    return ''.join(resultado)

def descifrar_vigenere(texto: str, clave: str) -> str:
    """Descifra texto cifrado con Vigenère."""
    texto = texto.upper()
    clave = clave.upper()
    resultado = []
    j = 0
    for char in texto:
        if char.isalpha():
            desplazamiento = ord(clave[j % len(clave)]) - ord('A')
            resultado.append(chr((ord(char) - ord('A') - desplazamiento) % 26 + ord('A')))
            j += 1
        else:
            resultado.append(char)
    return ''.join(resultado)

# Ejemplo
mensaje = 'ATACAR AL AMANECER'
clave = 'LIMON'
cifrado = cifrar_vigenere(mensaje, clave)
descifrado = descifrar_vigenere(cifrado, clave)

print(f'Mensaje:          {mensaje}')
print(f'Clave:            {clave}')
print(f'Clave repetida:   {(clave * (len(mensaje)//len(clave)+1))[:len(mensaje.replace(" ", ""))]}')
print(f'Texto cifrado:    {cifrado}')
print(f'Texto descifrado: {descifrado}')

### 7.4 Por qué Vigenère es más fuerte que César

Con César, la misma letra siempre se cifra de la misma manera → análisis de frecuencia directo.

Con Vigenère, una letra puede cifrarse de múltiples maneras dependiendo de su posición → el análisis de frecuencia simple no funciona directamente.

**Sin embargo**, no es invulnerable:
- Si se conoce la longitud de la clave, el mensaje se puede dividir en subgrupos, cada uno de los cuales es un cifrado César.
- El **Examen de Kasiski** permite encontrar la longitud de la clave.
- El **Índice de Coincidencia** (Friedman, 1922) también permite estimar la longitud.


In [ ]:
def indice_de_coincidencia(texto: str) -> float:
    """Calcula el Índice de Coincidencia (IC) de un texto.
    
    IC para inglés ≈ 0.065
    IC para texto aleatorio ≈ 0.038
    """
    texto = ''.join(c for c in texto.upper() if c.isalpha())
    n = len(texto)
    if n < 2:
        return 0.0
    frecuencias = collections.Counter(texto)
    numerador = sum(f * (f - 1) for f in frecuencias.values())
    denominador = n * (n - 1)
    return numerador / denominador

def estimar_longitud_clave_vigenere(ciphertext: str, max_len: int = 20) -> List[Tuple[int, float]]:
    """Estima la longitud de la clave Vigenère usando el Índice de Coincidencia."""
    texto = ''.join(c for c in ciphertext.upper() if c.isalpha())
    resultados = []
    for longitud in range(1, max_len + 1):
        # Dividir en 'longitud' subgrupos
        ic_promedio = 0.0
        for inicio in range(longitud):
            subgrupo = texto[inicio::longitud]
            ic_promedio += indice_de_coincidencia(subgrupo)
        ic_promedio /= longitud
        resultados.append((longitud, ic_promedio))
    return resultados

# Texto largo para demostración
texto_largo = """
The quick brown fox jumps over the lazy dog. 
Pack my box with five dozen liquor jugs. 
How vexingly quick daft zebras jump. 
The five boxing wizards jump quickly. 
Sphinx of black quartz judge my vow.
""" * 5

clave_secreta = 'CRYPTO'
ciphertext_largo = cifrar_vigenere(texto_largo, clave_secreta)

resultados_ic = estimar_longitud_clave_vigenere(ciphertext_largo)

print('Índice de Coincidencia por longitud de clave candidata:')
print(f'{"Longitud":>10} | {"IC Promedio":>12} | {"Interpretación":>20}')
print('-' * 50)
for longitud, ic in resultados_ic[:15]:
    interpretacion = '← CLAVE PROBABLE' if abs(ic - 0.065) < 0.01 else ''
    print(f'{longitud:>10} | {ic:>12.4f} | {interpretacion:>20}')
print(f'\nClave real usada: "{clave_secreta}" (longitud = {len(clave_secreta)})')

In [ ]:
# Visualización del IC por longitud de clave
longitudes = [r[0] for r in resultados_ic]
ics = [r[1] for r in resultados_ic]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(longitudes, ics, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axhline(y=0.065, color='green', linestyle='--', linewidth=1.5, label='IC del inglés ≈ 0.065')
ax.axhline(y=0.038, color='red', linestyle='--', linewidth=1.5, label='IC aleatorio ≈ 0.038')
ax.axvline(x=len(clave_secreta), color='orange', linestyle=':', linewidth=2,
           label=f'Longitud real de clave = {len(clave_secreta)}')
ax.set_xlabel('Longitud candidata de la clave')
ax.set_ylabel('IC promedio')
ax.set_title('Ataque Friedman: Estimación de longitud de clave Vigenère')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(longitudes)
plt.tight_layout()
plt.show()

---
## 8. Cifrado Playfair <a id='8-playfair'></a>

### 8.1 Historia

El cifrado Playfair fue inventado por **Charles Wheatstone** en 1854 pero popularizado y lleva el nombre de su amigo **Lord Playfair**, quien lo promovió ante el gobierno británico. Fue el primer cifrado práctico de **dígramas** (pares de letras). Fue usado por Gran Bretaña en la Primera y Segunda Guerra Mundial.

### 8.2 Construcción de la tabla Playfair

1. Elegir una clave (p. ej., `MONARCHY`).
2. Escribir las letras de la clave (sin repetir) seguidas del resto del alfabeto.
3. **I y J se combinan en una sola celda** (el alfabeto tiene 25 posiciones para una cuadrícula 5×5).
4. Organizar en una cuadrícula 5×5.

### 8.3 Reglas de cifrado

1. **Preparación:** Dividir el plaintext en pares (dígramas). Si las letras de un par son iguales, insertar una letra de relleno (p. ej., `X`). Si el texto tiene longitud impar, añadir `X` al final. I → J (o viceversa).
2. **Par en la misma fila:** Reemplazar cada letra por la que está a su derecha (circular).
3. **Par en la misma columna:** Reemplazar cada letra por la que está debajo (circular).
4. **Par en posiciones diferentes:** Cada letra se reemplaza por la letra en su misma fila pero en la columna de la otra letra (rectángulo).


In [ ]:
class CifradoPlayfair:
    """Implementación del cifrado Playfair."""

    def __init__(self, clave: str):
        self.tabla, self.pos = self._construir_tabla(clave)

    def _construir_tabla(self, clave: str) -> Tuple[List[List[str]], Dict[str, Tuple[int, int]]]:
        clave = clave.upper().replace('J', 'I')
        vistos = []
        for c in clave:
            if c.isalpha() and c not in vistos:
                vistos.append(c)
        for c in string.ascii_uppercase:
            if c != 'J' and c not in vistos:
                vistos.append(c)
        tabla = [vistos[i*5:(i+1)*5] for i in range(5)]
        pos = {vistos[i]: (i // 5, i % 5) for i in range(25)}
        return tabla, pos

    def _preparar_texto(self, texto: str) -> List[str]:
        texto = texto.upper().replace('J', 'I')
        texto = ''.join(c for c in texto if c.isalpha())
        resultado = []
        i = 0
        while i < len(texto):
            a = texto[i]
            if i + 1 < len(texto):
                b = texto[i + 1]
                if a == b:
                    resultado.append(a + 'X')
                    i += 1
                else:
                    resultado.append(a + b)
                    i += 2
            else:
                resultado.append(a + 'X')
                i += 1
        return resultado

    def _cifrar_digrama(self, a: str, b: str, modo: int) -> str:
        """modo: +1 para cifrar, -1 para descifrar."""
        ra, ca = self.pos[a]
        rb, cb = self.pos[b]
        if ra == rb:  # misma fila
            return self.tabla[ra][(ca + modo) % 5] + self.tabla[rb][(cb + modo) % 5]
        elif ca == cb:  # misma columna
            return self.tabla[(ra + modo) % 5][ca] + self.tabla[(rb + modo) % 5][cb]
        else:  # rectángulo
            return self.tabla[ra][cb] + self.tabla[rb][ca]

    def cifrar(self, texto: str) -> str:
        digramas = self._preparar_texto(texto)
        return ' '.join(self._cifrar_digrama(d[0], d[1], +1) for d in digramas)

    def descifrar(self, ciphertext: str) -> str:
        texto = ciphertext.replace(' ', '')
        digramas = [texto[i:i+2] for i in range(0, len(texto), 2)]
        return ''.join(self._cifrar_digrama(d[0], d[1], -1) for d in digramas)

    def mostrar_tabla(self):
        print('Tabla Playfair:')
        print('┌───┬───┬───┬───┬───┐')
        for i, fila in enumerate(self.tabla):
            print('│ ' + ' │ '.join(fila) + ' │')
            if i < 4:
                print('├───┼───┼───┼───┼───┤')
        print('└───┴───┴───┴───┴───┘')


# Demostración
pf = CifradoPlayfair('MONARCHY')
pf.mostrar_tabla()

mensaje = 'HIDE THE GOLD IN THE TREE STUMP'
cifrado = pf.cifrar(mensaje)
descifrado = pf.descifrar(cifrado.replace(' ', ''))

print(f'\nMensaje:          {mensaje}')
print(f'Texto cifrado:    {cifrado}')
print(f'Texto descifrado: {descifrado}')

### 8.4 Seguridad del Playfair

**Fortalezas vs. César/Vigenère:**
- Opera sobre **dígramas** en lugar de letras individuales → mayor resistencia al análisis de frecuencia simple.
- Espacio de claves mucho mayor: $25! \approx 1.55 \times 10^{25}$ posibles tablas.

**Debilidades:**
- Sigue siendo un cifrado de sustitución → vulnerable al análisis de frecuencia de dígramas.
- Las letras del mismo dígrama se mapean a una relación bien definida (la propiedad del rectángulo revela la estructura).
- Con suficiente texto cifrado, se puede romper.
- Fácil de detectar cuando ocurre (`XX` u otros patrones de relleno).


---
## 9. Análisis de Frecuencia <a id='9-frecuencia'></a>

### 9.1 Principio

El **análisis de frecuencia** es la técnica de criptoanálisis más antigua y poderosa contra cifrados de sustitución. Fue descrita por **Al-Kindi** alrededor del año 850 d.C.

**Principio:** En cualquier idioma, las letras no se distribuyen uniformemente. Si un cifrado preserva la frecuencia de los caracteres (como el cifrado César), entonces el ciphertext tendrá las mismas frecuencias que el idioma original, pero desplazadas.

### 9.2 Frecuencias del español

En español, las letras más frecuentes son:

| Letra | Freq. | Letra | Freq. |
|-------|-------|-------|-------|
| E | 13.7% | S | 7.2% |
| A | 11.5% | N | 6.7% |
| O | 8.7%  | R | 6.9% |
| I | 6.3%  | L | 4.9% |

### 9.3 Método de ataque

1. Contar la frecuencia de cada letra en el ciphertext.
2. Ordenar de mayor a menor frecuencia.
3. Mapear las letras más frecuentes del ciphertext a las letras más frecuentes del idioma.
4. Usar contexto lingüístico para refinar el mapeo.


In [ ]:
# Frecuencias de letras en español e inglés
FREQ_ESPANOL = {
    'A': 11.53, 'B': 1.42, 'C': 4.68, 'D': 5.86, 'E': 13.68, 'F': 0.69,
    'G': 1.01, 'H': 0.70, 'I': 6.25, 'J': 0.44, 'K': 0.01, 'L': 4.97,
    'M': 3.15, 'N': 6.71, 'O': 8.68, 'P': 2.51, 'Q': 0.88, 'R': 6.87,
    'S': 7.98, 'T': 4.63, 'U': 3.93, 'V': 0.90, 'W': 0.01, 'X': 0.22,
    'Y': 0.90, 'Z': 0.52
}

FREQ_INGLES = {
    'A': 8.17, 'B': 1.49, 'C': 2.78, 'D': 4.25, 'E': 12.70, 'F': 2.23,
    'G': 2.02, 'H': 6.09, 'I': 6.97, 'J': 0.15, 'K': 0.77, 'L': 4.03,
    'M': 2.41, 'N': 6.75, 'O': 7.51, 'P': 1.93, 'Q': 0.10, 'R': 5.99,
    'S': 6.33, 'T': 9.06, 'U': 2.76, 'V': 0.98, 'W': 2.36, 'X': 0.15,
    'Y': 1.97, 'Z': 0.07
}

def frecuencia_letras(texto: str) -> Dict[str, float]:
    """Calcula la frecuencia porcentual de cada letra en un texto."""
    texto = ''.join(c for c in texto.upper() if c.isalpha())
    if not texto:
        return {}
    conteo = collections.Counter(texto)
    total = len(texto)
    return {letra: (conteo.get(letra, 0) / total) * 100
            for letra in string.ascii_uppercase}

def visualizar_frecuencias(texto: str, titulo: str, freq_referencia: Dict = None):
    """Visualiza las frecuencias de letras de un texto."""
    freq = frecuencia_letras(texto)
    letras = list(string.ascii_uppercase)
    valores = [freq.get(l, 0) for l in letras]

    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(letras))
    ax.bar(x, valores, color='steelblue', alpha=0.8, label='Texto analizado')

    if freq_referencia:
        ref_vals = [freq_referencia.get(l, 0) for l in letras]
        ax.plot(x, ref_vals, 'r--o', markersize=4, linewidth=1.5,
                label='Frecuencia de referencia (idioma)')

    ax.set_xticks(x)
    ax.set_xticklabels(letras)
    ax.set_xlabel('Letra')
    ax.set_ylabel('Frecuencia (%)')
    ax.set_title(titulo)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

# Visualizar frecuencias del inglés
texto_muestra = """
To be or not to be that is the question whether tis nobler in the mind to 
suffer the slings and arrows of outrageous fortune or to take arms against 
a sea of troubles and by opposing end them to die to sleep no more and by 
a sleep to say we end the heartache and the thousand natural shocks that 
flesh is heir to tis a consummation devoutly to be wished to die to sleep.
"""

visualizar_frecuencias(texto_muestra, 'Frecuencias en texto inglés (Hamlet)', FREQ_INGLES)

In [ ]:
# Demostración: Las frecuencias se PRESERVAN con César
clave_desconocida = 7
ciphertext_hamlet = cifrar_cesar(texto_muestra, clave_desconocida)

print(f'Texto cifrado con César k={clave_desconocida}:')
print(ciphertext_hamlet[:200])

visualizar_frecuencias(
    ciphertext_hamlet,
    f'Frecuencias del texto cifrado con César (k={clave_desconocida}) — las frecuencias se desplazan',
    FREQ_INGLES
)

---
## 10. Práctica: Implementar y Romper Cifrados Clásicos <a id='10-practica'></a>

### 10.1 Ataque de Análisis de Frecuencia al Cifrado César


In [ ]:
def romper_cesar_frecuencia(ciphertext: str, idioma: str = 'en') -> Tuple[int, str, float]:
    """Rompe el cifrado César usando análisis de frecuencia.
    
    Devuelve: (clave_estimada, plaintext_estimado, puntuación)
    """
    freq_ref = FREQ_INGLES if idioma == 'en' else FREQ_ESPANOL
    letras_ref = sorted(freq_ref, key=freq_ref.get, reverse=True)

    freq_cipher = frecuencia_letras(ciphertext)
    letra_mas_freq_cipher = max(freq_cipher, key=freq_cipher.get)

    # La letra más frecuente en el cipher probablemente corresponde
    # a la letra más frecuente en el idioma
    letra_mas_freq_ref = letras_ref[0]  # 'E' para inglés

    clave_estimada = (ord(letra_mas_freq_cipher) - ord(letra_mas_freq_ref)) % 26
    plaintext_estimado = descifrar_cesar(ciphertext, clave_estimada)

    # Puntuación: correlación con frecuencias del idioma
    freq_plain = frecuencia_letras(plaintext_estimado)
    puntuacion = sum(freq_plain.get(l, 0) * freq_ref.get(l, 0)
                     for l in string.ascii_uppercase)

    return clave_estimada, plaintext_estimado, puntuacion

def romper_cesar_exhaustivo(ciphertext: str, idioma: str = 'en') -> List[Tuple[int, str, float]]:
    """Prueba todas las claves posibles y las ordena por puntuación de idioma."""
    freq_ref = FREQ_INGLES if idioma == 'en' else FREQ_ESPANOL
    resultados = []
    for k in range(26):
        plaintext = descifrar_cesar(ciphertext, k)
        freq = frecuencia_letras(plaintext)
        puntuacion = sum(freq.get(l, 0) * freq_ref.get(l, 0)
                         for l in string.ascii_uppercase)
        resultados.append((k, plaintext, puntuacion))
    return sorted(resultados, key=lambda x: x[2], reverse=True)


# Cifrar un texto secreto con una clave desconocida
mensaje_secreto = """
In the beginning was the word and the word was with god and the word was god. 
All things were made through him and without him was not any thing made that 
was made in him was life and the life was the light of men.
"""

clave_secreta = 19  # ¡Esta clave está oculta para el ataque!
ciphertext_secreto = cifrar_cesar(mensaje_secreto, clave_secreta)

print('TEXTO CIFRADO (interceptado):')
print(ciphertext_secreto[:200])
print('\n' + '='*60)

# Ataque
clave_est, plain_est, score = romper_cesar_frecuencia(ciphertext_secreto)
print(f'\n🔍 Ataque por análisis de frecuencia:')
print(f'Clave estimada: k = {clave_est}')
print(f'Plaintext recuperado:\n{plain_est[:200]}')
print(f'\n✅ Clave real: {clave_secreta} | Clave estimada: {clave_est} | ¿Correcto?: {clave_est == clave_secreta}')

In [ ]:
# Mostrar todas las claves ordenadas por puntuación
print('Ranking de claves por correlación con idioma inglés:')
print(f'{"Rank":>5} | {"Clave":>5} | {"Puntuación":>12} | {"Primeros 50 chars"}')
print('-' * 80)
for rank, (k, plain, score) in enumerate(romper_cesar_exhaustivo(ciphertext_secreto)[:10], 1):
    plain_limpio = ''.join(plain.split())[:50]
    print(f'{rank:>5} | {k:>5} | {score:>12.4f} | {plain_limpio}')

print(f'\n🏆 La clave con mayor puntuación es la correcta!')

### 10.2 Ataque al Cifrado Vigenère

El ataque al cifrado Vigenère requiere dos pasos:
1. **Determinar la longitud de la clave** (Kasiski o Friedman)
2. **Romper cada subgrupo** como un cifrado César individual


In [ ]:
def kasiski_examination(ciphertext: str, min_len: int = 3, max_len: int = 6) -> Dict[int, int]:
    """Realiza el examen de Kasiski para encontrar repeticiones en el ciphertext.
    Devuelve un diccionario con distancias entre repeticiones y sus frecuencias.
    """
    texto = ''.join(c for c in ciphertext.upper() if c.isalpha())
    espaciados = collections.Counter()

    for longitud in range(min_len, max_len + 1):
        for i in range(len(texto) - longitud):
            fragmento = texto[i:i+longitud]
            # Buscar repeticiones
            j = i + longitud
            while j <= len(texto) - longitud:
                idx = texto.find(fragmento, j)
                if idx != -1:
                    distancia = idx - i
                    # Factores de la distancia (candidatos a longitud de clave)
                    for f in range(2, distancia + 1):
                        if distancia % f == 0:
                            espaciados[f] += 1
                    j = idx + 1
                else:
                    break

    return dict(espaciados.most_common(15))

def romper_vigenere(ciphertext: str, longitud_clave: int, idioma: str = 'en') -> str:
    """Rompe el cifrado Vigenère conociendo la longitud de la clave."""
    freq_ref = FREQ_INGLES if idioma == 'en' else FREQ_ESPANOL
    texto = ''.join(c for c in ciphertext.upper() if c.isalpha())
    clave_recuperada = []

    for inicio in range(longitud_clave):
        subgrupo = texto[inicio::longitud_clave]
        # Encontrar la mejor clave para este subgrupo (César)
        mejor_k, _, _ = romper_cesar_frecuencia(subgrupo, idioma)
        clave_recuperada.append(chr(mejor_k + ord('A')))

    clave = ''.join(clave_recuperada)
    plaintext = descifrar_vigenere(ciphertext, clave)
    return clave, plaintext


# Texto largo en inglés para demostración
texto_poe = """
It was many and many a year ago in a kingdom by the sea that a maiden there 
lived whom you may know by the name of Annabel Lee and this maiden she lived 
with no other thought than to love and be loved by me I was a child and she 
was a child in this kingdom by the sea but we loved with a love that was more 
than love I and my Annabel Lee with a love that the winged seraphs of Heaven 
coveted her and me and this was the reason that long ago in this kingdom by the 
sea a wind blew out of a cloud chilling my beautiful Annabel Lee so that her 
highborn kinsmen came and bore her away from me to shut her up in a sepulchre 
in this kingdom by the sea.
"""

clave_vigenere = 'EDGAR'
ct_poe = cifrar_vigenere(texto_poe, clave_vigenere)

print('Texto cifrado (primeros 200 chars):')
print(ct_poe[:200])
print('\n' + '='*60)

# Paso 1: Estimar longitud con IC
resultados_ic = estimar_longitud_clave_vigenere(ct_poe, max_len=15)
longitud_estimada = max(resultados_ic, key=lambda x: x[1])[0]
print(f'\nLongitud estimada por IC: {longitud_estimada}')

# Paso 2: Romper con la longitud correcta
clave_encontrada, plaintext_recuperado = romper_vigenere(ct_poe, len(clave_vigenere))
print(f'Clave recuperada: {clave_encontrada}')
print(f'Clave real:       {clave_vigenere}')
print(f'\nPlaintext recuperado (primeros 200 chars):')
print(plaintext_recuperado[:200])

In [ ]:
# Examen de Kasiski
resultados_kasiski = kasiski_examination(ct_poe)
print('Examen de Kasiski — factores más comunes de distancias entre repeticiones:')
print(f'{"Factor":>8} | {"Frecuencia":>12}')
print('-' * 25)
for factor, freq in sorted(resultados_kasiski.items(), key=lambda x: x[1], reverse=True)[:10]:
    marcador = ' ← clave real' if factor == len(clave_vigenere) else ''
    print(f'{factor:>8} | {freq:>12}{marcador}')
print(f'\nLongitud real de la clave: {len(clave_vigenere)}')

### 10.3 One-Time Pad: El único cifrado teóricamente perfecto


In [ ]:
def otp_cifrar(mensaje: str) -> Tuple[bytes, bytes]:
    """One-Time Pad: cifrado con XOR usando clave aleatoria.
    Devuelve (ciphertext_bytes, clave_bytes).
    """
    msg_bytes = mensaje.encode('utf-8')
    clave_bytes = secrets.token_bytes(len(msg_bytes))  # Clave verdaderamente aleatoria
    ciphertext = bytes(a ^ b for a, b in zip(msg_bytes, clave_bytes))
    return ciphertext, clave_bytes

def otp_descifrar(ciphertext: bytes, clave: bytes) -> str:
    """Descifra un mensaje OTP."""
    plaintext_bytes = bytes(a ^ b for a, b in zip(ciphertext, clave))
    return plaintext_bytes.decode('utf-8')

mensaje = 'ATACAR AL AMANECER'
ct_otp, clave_otp = otp_cifrar(mensaje)
pt_otp = otp_descifrar(ct_otp, clave_otp)

print(f'Mensaje original:  {mensaje}')
print(f'Clave OTP (hex):   {clave_otp.hex()}')
print(f'Ciphertext (hex):  {ct_otp.hex()}')
print(f'Descifrado:        {pt_otp}')

print('\n--- Demostración: Maleabilidad del OTP (ambigüedad perfecta) ---')
# Con OTP, cualquier plaintext es igualmente probable para el mismo ciphertext
mensaje_falso = 'RETIRAR AL MEDIODI'
assert len(mensaje_falso) == len(mensaje), 'Misma longitud necesaria'

clave_falsa = bytes(ct ^ pt for ct, pt in zip(ct_otp, mensaje_falso.encode('utf-8')))
verificacion = otp_descifrar(ct_otp, clave_falsa)

print(f'Con clave_falsa (hex): {clave_falsa.hex()}')
print(f'Se descifra como:      {verificacion}')
print('\n⚠️  El mismo ciphertext puede descifrar a CUALQUIER mensaje del mismo largo.')
print('Esto es "ambigüedad perfecta" — sin la clave correcta, es imposible saber cuál es el real.')

### 10.4 Desafío Práctico: ¿Puedes romper este mensaje?


In [ ]:
# ─── DESAFÍO ───
# Este mensaje fue cifrado con un método que aprendiste en esta clase.
# ¿Puedes descifrar el mensaje y encontrar la clave?

mensaje_desafio = """
AOPZPZVUJPZMAHKVCPNLULYL
HWSPHRLSHZOLYYHTPLUHZKVUKL
HWYLUKPZALZSHZILZAJPWYHZ
"""

print('🔒 MENSAJE CIFRADO (interceptado):')
print(mensaje_desafio)
print('Pista: Es uno de los cifrados vistos en esta clase.')
print('Pista 2: La clave es un número entre 1 y 25.')
print('\n' + '─'*50)
print('Intenta resolverlo tú mismo antes de ver la solución...')
print('─'*50)

In [ ]:
# ─── SOLUCIÓN DEL DESAFÍO ───
# Descomenta la siguiente línea para ver la solución:

# print('Solución:')
# for k, plain, score in romper_cesar_exhaustivo(mensaje_desafio.replace('\n', ''))[:3]:
#     print(f'k={k}: {plain}')

print('(Solución oculta — descomenta el código para verla)')

### 10.5 Análisis comparativo de los tres cifrados clásicos


In [ ]:
texto_referencia = """
The study of cryptography and cryptanalysis is fundamental to understanding 
modern information security. Classical ciphers, while no longer secure by 
modern standards, illustrate the principles that underlie contemporary systems.
"""

# Cifrar el mismo texto con los tres métodos
ct_cesar = cifrar_cesar(texto_referencia, 13)
ct_vigenere = cifrar_vigenere(texto_referencia, 'CRYPTO')
ct_playfair = CifradoPlayfair('SECURITY').cifrar(texto_referencia)

# Calcular IC para cada uno
ic_original = indice_de_coincidencia(texto_referencia)
ic_cesar = indice_de_coincidencia(ct_cesar)
ic_vigenere = indice_de_coincidencia(ct_vigenere)
ic_playfair = indice_de_coincidencia(ct_playfair)

print('Comparación de Índice de Coincidencia:')
print(f'  Texto original (inglés): IC = {ic_original:.4f} (esperado ≈ 0.065)')
print(f'  César (ROT-13):          IC = {ic_cesar:.4f} (debería ser ≈ 0.065 — preserva frecuencias)')
print(f'  Vigenère (clave=CRYPTO): IC = {ic_vigenere:.4f} (debería ser < 0.065)')
print(f'  Playfair (clave=SECURITY): IC = {ic_playfair:.4f}')

# Visualización comparativa
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
datos = [
    (texto_referencia, 'Texto original', axes[0,0]),
    (ct_cesar, 'Cifrado César (ROT-13)', axes[0,1]),
    (ct_vigenere, 'Cifrado Vigenère (clave=CRYPTO)', axes[1,0]),
    (ct_playfair, 'Cifrado Playfair (clave=SECURITY)', axes[1,1]),
]

letras = list(string.ascii_uppercase)
x = np.arange(len(letras))
ref_vals = [FREQ_INGLES.get(l, 0) for l in letras]

for texto, titulo, ax in datos:
    freq = frecuencia_letras(texto)
    valores = [freq.get(l, 0) for l in letras]
    ax.bar(x, valores, color='steelblue', alpha=0.7)
    ax.plot(x, ref_vals, 'r--', linewidth=1.5, alpha=0.7, label='Ref. inglés')
    ax.set_xticks(x)
    ax.set_xticklabels(letras, fontsize=8)
    ic = indice_de_coincidencia(texto)
    ax.set_title(f'{titulo}\nIC = {ic:.4f}', fontsize=11)
    ax.set_ylabel('Frecuencia (%)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Análisis de Frecuencia: Comparación de Cifrados Clásicos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Resumen y Conclusiones <a id='11-resumen'></a>

### Lo que aprendimos hoy

| Tema | Conceptos clave |
|------|-----------------|
| **Historia** | Criptografía antigua → Al-Kindi → Kerckhoffs → Shannon → Criptografía moderna |
| **Principios de Kerckhoffs** | La seguridad reside en la **clave**, no en el secreto del algoritmo |
| **Entropía** | Mide incertidumbre; alta entropía = más seguro; $H(X) = -\sum p \log_2 p$ |
| **Aleatoriedad** | TRNG > CSPRNG >> PRNG para aplicaciones criptográficas |
| **Cifrado César** | Sustitución monoalfabética; 25 claves; trivialmente rompible |
| **Cifrado Vigenère** | Polialfabético; más fuerte pero vulnerable a Kasiski/Friedman |
| **Cifrado Playfair** | Opera sobre dígramas; 5×5; mayor resistencia al análisis de frecuencia |
| **Análisis de Frecuencia** | Explota la distribución no uniforme de letras en los idiomas naturales |

### Lecciones fundamentales

1. **Ningún cifrado clásico es seguro** por estándares modernos.
2. **La seguridad a través de la oscuridad no funciona** — los algoritmos deben ser públicos.
3. **La aleatoriedad es fundamental** — una clave débil compromete cualquier cifrado.
4. **El análisis de frecuencia** es una herramienta poderosa contra cifrados de sustitución.
5. **La entropía** es la medida formal de seguridad de una clave.

### Hacia la criptografía moderna

Los cifrados clásicos fallan porque:
- El espacio de claves es pequeño.
- No son seguros contra el modelo de adversario CPA/CCA.
- No satisfacen la definición formal de **seguridad semántica**.

La criptografía moderna (AES, RSA, curvas elípticas) supera estas limitaciones con:
- Confusión y difusión (principios de Shannon)
- Fundamentos matemáticos en problemas computacionalmente difíciles
- Definiciones formales de seguridad probablemente seguras

### Próxima clase

- **Clase 2:** Cifrados de bloque y flujo modernos — DES, AES, modos de operación (ECB, CBC, CTR, GCM).


In [ ]:
# Cuadro comparativo final
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

columnas = ['Cifrado', 'Tipo', 'Espacio de claves', 'IC preservado', 'Ruptura clásica', 'Año']
filas = [
    ['César', 'Sustitución monoalf.', '25 (≈4.6 bits)', 'Sí', 'Fuerza bruta / Frec.', '~100 a.C.'],
    ['Vigenère', 'Sustitución polialf.', '26^n (n=long. clave)', 'Parcial', 'Kasiski + Frec.', '1553'],
    ['Playfair', 'Sustitución dígramas', '~25! ≈ 10^25', 'No (dígramas)', 'Frec. dígramas', '1854'],
    ['OTP', 'XOR (flujo)', '∞ (perfecta)', 'No', 'Imposible (teórico)', '1882'],
]

tabla = ax.table(
    cellText=filas,
    colLabels=columnas,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(11)

# Estilo encabezados
for j in range(len(columnas)):
    tabla[0, j].set_facecolor('#2c3e50')
    tabla[0, j].set_text_props(color='white', fontweight='bold')

# Alternar colores de filas
colores = ['#ecf0f1', '#ffffff', '#ecf0f1', '#ffffff']
for i, color in enumerate(colores, 1):
    for j in range(len(columnas)):
        tabla[i, j].set_facecolor(color)

ax.set_title('Resumen: Cifrados Clásicos Estudiados', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

---
## 12. Referencias <a id='12-referencias'></a>

### Libros fundamentales

1. **Schneier, B.** (1996). *Applied Cryptography: Protocols, Algorithms, and Source Code in C* (2nd ed.). Wiley.

2. **Stinson, D. R.** (2005). *Cryptography: Theory and Practice* (3rd ed.). Chapman & Hall/CRC.

3. **Katz, J., & Lindell, Y.** (2020). *Introduction to Modern Cryptography* (3rd ed.). CRC Press.

4. **Paar, C., & Pelzl, J.** (2010). *Understanding Cryptography: A Textbook for Students and Practitioners*. Springer. [Material online gratuito: https://www.crypto-textbook.com/]

5. **Menezes, A., van Oorschot, P., & Vanstone, S.** (1996). *Handbook of Applied Cryptography*. CRC Press. [Disponible gratis en: https://cacr.uwaterloo.ca/hac/]

### Artículos originales

6. **Shannon, C. E.** (1949). Communication Theory of Secrecy Systems. *Bell System Technical Journal*, 28(4), 656–715.

7. **Shannon, C. E.** (1948). A Mathematical Theory of Communication. *Bell System Technical Journal*, 27(3), 379–423.

8. **Kerckhoffs, A.** (1883). La cryptographie militaire. *Journal des Sciences Militaires*, 9, 5–38.

9. **Diffie, W., & Hellman, M.** (1976). New Directions in Cryptography. *IEEE Transactions on Information Theory*, 22(6), 644–654.

### Recursos en línea

10. **Khan Academy** — Criptografía: https://www.khanacademy.org/computing/computer-science/cryptography

11. **Coursera — Cryptography I** (Dan Boneh, Stanford): https://www.coursera.org/learn/crypto

12. **CrypTool 2** — Herramienta visual para criptografía: https://www.cryptool.org/en/ct2/

13. **dCode.fr** — Herramientas de criptoanálisis online: https://www.dcode.fr/

14. **Crypto101** (Ivan Ristic): https://crypto101.io/ — Libro gratuito introductorio.

15. **Al-Kindi** (≈850 d.C.). *Risalah fi Istikhraj al-Mu'amma* (Manuscrito para el Descifrado de Mensajes Cifrados). Traducción al inglés disponible en el Museo Árabe de Ciencia y Tecnología, Dubái.

### Para profundizar

16. **NIST SP 800-22** — Pruebas estadísticas para generadores de números aleatorios y pseudoaleatorios: https://csrc.nist.gov/publications/detail/sp/800-22/rev-1a/final

17. **Friedman, W.** (1922). The Index of Coincidence and Its Applications in Cryptography. Riverbank Laboratories.

18. **Singh, S.** (1999). *The Code Book: The Science of Secrecy from Ancient Egypt to Quantum Cryptography*. Doubleday. [Lectura divulgativa altamente recomendada]
